# VoiceForge - TTS + Video Engine (Kaggle)

This notebook runs the complete VoiceForge inference pipeline:
- **Fish Audio S2 Pro** for voice cloning + TTS
- **Remotion** for video rendering with caption highlighting

Runs on Kaggle with GPU acceleration. Exposes a FastAPI server via ngrok.

In [ ]:
# Cell 1: Install All Dependencies
!git clone https://github.com/fishaudio/fish-speech.git
!cd fish-speech && pip install -e .[cu129]

# Video processing dependencies
!pip install remotion @remotion/cli @remotion/captions
!pip install fastapi uvicorn pyngrok httpx numpy soundfile Pillow
!apt-get update && apt-get install -y ffmpeg

In [ ]:
# Cell 2: Setup Credentials & Download Model Weights
import os

# ---- Paste your tokens below ----
HF_TOKEN = ''          # HuggingFace token (https://huggingface.co/settings/tokens)
NGROK_AUTH_TOKEN = ''  # Ngrok auth token (https://dashboard.ngrok.com/get-started/your-authtoken)

# Auto-detect from Kaggle secrets if tokens not set above
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not NGROK_AUTH_TOKEN:
    NGROK_AUTH_TOKEN = os.environ.get('NGROK_AUTH_TOKEN', '')

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['NGROK_AUTH_TOKEN'] = NGROK_AUTH_TOKEN

print(f'HF_TOKEN: {"SET" if HF_TOKEN else "MISSING - paste in Cell 2"}')
print(f'NGROK_AUTH_TOKEN: {"SET" if NGROK_AUTH_TOKEN else "MISSING - paste in Cell 2"}')

!huggingface-cli login --token {HF_TOKEN}
!cd fish-speech && hf download fishaudio/s2-pro --local-dir checkpoints/s2-pro

In [ ]:
# Cell 3: TTS Inference Functions
import subprocess
import numpy as np
import soundfile as sf
from pathlib import Path
import io
import os
import sys

# 1. Force Unix commands to always answer 'yes'
os.environ["DEBIAN_FRONTEND"] = "noninteractive"

# 2. Automatically feed 'y' into any Python input() prompts
sys.stdin = io.StringIO("y\n" * 100)

print("Setup complete. Automations active.")

WORK_DIR = Path('/kaggle/working/voiceforge')
WORK_DIR.mkdir(exist_ok=True)

FISH_DIR = Path('/kaggle/working/fish-speech')
CHECKPOINT = FISH_DIR / 'checkpoints/s2-pro'


def extract_voice_tokens(audio_path: str) -> dict:
    """Extract VQ tokens from reference audio using Fish Audio codec."""
    output_npy = WORK_DIR / 'ref_tokens.npy'
    
    result = subprocess.run([
        'python', str(FISH_DIR / 'fish_speech/models/dac/inference.py'),
        '-i', audio_path,
        '--checkpoint-path', str(CHECKPOINT / 'codec.pth'),
        '--output', str(output_npy)
    ], capture_output=True, text=True, cwd=str(FISH_DIR))
    
    if result.returncode != 0:
        raise Exception(f'Token extraction failed: {result.stderr}')
    
    return {'tokens_path': str(output_npy), 'status': 'success'}


def generate_tts(text: str, tokens_path: str, chapter_num: int) -> dict:
    """Generate TTS audio from text using cloned voice."""
    codes_path = WORK_DIR / f'codes_{chapter_num}.npy'
    output_wav = WORK_DIR / f'chapter_{chapter_num}.wav'
    
    # Step 1: Generate semantic tokens
    subprocess.run([
        'python', str(FISH_DIR / 'fish_speech/models/text2semantic/inference.py'),
        '--text', text,
        '--prompt-tokens', tokens_path,
        '--output', str(codes_path)
    ], capture_output=True, text=True, cwd=str(FISH_DIR))
    
    # Step 2: Decode to audio
    subprocess.run([
        'python', str(FISH_DIR / 'fish_speech/models/dac/inference.py'),
        '-i', str(codes_path),
        '--output', str(output_wav)
    ], capture_output=True, text=True, cwd=str(FISH_DIR))
    
    return {
        'status': 'success',
        'audio_path': str(output_wav),
        'chapter_number': chapter_num
    }


def batch_tts(chapters: list[dict], tokens_path: str) -> list[dict]:
    """Generate TTS for multiple chapters."""
    results = []
    for chapter in chapters:
        result = generate_tts(
            text=chapter['text'],
            tokens_path=tokens_path,
            chapter_num=chapter['number']
        )
        results.append(result)
    return results

In [ ]:
# Cell 4: Video Rendering with Remotion
import json
import tempfile
from pathlib import Path


def render_video(
    audio_path: str,
    captions: list[dict],
    aspect_ratio: str = '16:9',
    background_color: str = '#ffffff',
    caption_style: str = 'tiktok',
    title: str = 'Chapter 1',
    subtitle: str = ''
) -> dict:
    """Render audiobook video with Remotion using GPU acceleration."""
    
    # Parse aspect ratio
    ratios = {
        '16:9': (1920, 1080),
        '9:16': (1080, 1920),
        '1:1': (1080, 1080),
        '4:5': (1080, 1350),
    }
    width, height = ratios.get(aspect_ratio, (1920, 1080))
    
    # Create Remotion input props
    props = {
        'audioUrl': audio_path,
        'captions': captions,
        'aspectRatio': aspect_ratio,
        'backgroundColor': background_color,
        'captionStyle': caption_style,
        'title': title,
        'subtitle': subtitle,
    }
    
    props_file = WORK_DIR / 'remotion_props.json'
    with open(props_file, 'w') as f:
        json.dump(props, f)
    
    output_video = WORK_DIR / f'video_{aspect_ratio.replace(":", "-")}.mp4'
    
    # Render with Remotion CLI (uses GPU)
    comp_id = f'AudiobookVideo-{aspect_ratio.replace(":", "-")}'
    result = subprocess.run([
        'npx', 'remotion', 'render',
        'src/index.ts',
        comp_id,
        str(output_video),
        '--props', str(props_file),
        '--gl', 'angle',  # GPU acceleration
    ], capture_output=True, text=True, cwd=str(WORK_DIR / 'video-service'))
    
    if result.returncode != 0:
        raise Exception(f'Video render failed: {result.stderr}')
    
    return {
        'status': 'success',
        'video_path': str(output_video),
        'aspect_ratio': aspect_ratio
    }


def concat_chapters(video_paths: list[str], output_path: str) -> dict:
    """Concatenate multiple chapter videos into one."""
    # Create concat file
    concat_file = WORK_DIR / 'concat.txt'
    with open(concat_file, 'w') as f:
        for path in video_paths:
            f.write(f"file '{path}'\n")
    
    # Use ffmpeg to concatenate
    result = subprocess.run([
        'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
        '-i', str(concat_file),
        '-c', 'copy', output_path
    ], capture_output=True, text=True)
    
    if result.returncode != 0:
        raise Exception(f'Concat failed: {result.stderr}')
    
    return {'status': 'success', 'video_path': output_path}

In [ ]:
# Cell 5: Start Fish Audio S2 API Server (background)
import threading
import time

def start_fish_server():
    subprocess.run([
        'python', str(FISH_DIR / 'tools/api_server.py'),
        '--llama-checkpoint-path', str(CHECKPOINT),
        '--decoder-checkpoint-path', str(CHECKPOINT / 'codec.pth'),
        '--decoder-config-name', 'modded_dac_vq',
        '--listen', '0.0.0.0:8888',
        '--compile'
    ])

fish_thread = threading.Thread(target=start_fish_server, daemon=True)
fish_thread.start()
time.sleep(15)
print('Fish Audio S2 server started on port 8888')

In [ ]:
# Cell 6: Unified VoiceForge API Server
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import FileResponse, JSONResponse
import httpx
import uuid

app = FastAPI(title='VoiceForge Engine')

# Store for voice tokens
voice_store = {}


# ---- Voice Cloning Endpoints ----

@app.post('/api/voice/extract-tokens')
async def api_extract_tokens(
    reference_audio: UploadFile = File(...),
    name: str = Form('default')
):
    """Extract voice tokens from reference audio."""
    audio_path = WORK_DIR / f'ref_{uuid.uuid4()}.wav'
    with open(audio_path, 'wb') as f:
        content = await reference_audio.read()
        f.write(content)
    
    result = extract_voice_tokens(str(audio_path))
    voice_id = str(uuid.uuid4())
    voice_store[voice_id] = result['tokens_path']
    
    return {'voice_id': voice_id, 'status': 'success'}


# ---- TTS Endpoints ----

@app.post('/api/tts/generate')
async def api_generate_tts(
    text: str = Form(...),
    voice_id: str = Form(...),
    chapter_number: int = Form(1)
):
    """Generate TTS audio."""
    if voice_id not in voice_store:
        raise HTTPException(404, 'Voice not found')
    
    tokens_path = voice_store[voice_id]
    result = generate_tts(text, tokens_path, chapter_number)
    
    return {
        'audio_path': result['audio_path'],
        'chapter_number': chapter_number,
        'status': 'success'
    }


@app.post('/api/tts/batch')
async def api_batch_tts(
    chapters_json: str = Form(...),
    voice_id: str = Form(...)
):
    """Batch generate TTS for multiple chapters."""
    if voice_id not in voice_store:
        raise HTTPException(404, 'Voice not found')
    
    chapters = json.loads(chapters_json)
    tokens_path = voice_store[voice_id]
    results = batch_tts(chapters, tokens_path)
    
    return {'results': results, 'status': 'success'}


# ---- Video Endpoints ----

@app.post('/api/video/render')
async def api_render_video(
    audio_path: str = Form(...),
    captions_json: str = Form('[]'),
    aspect_ratio: str = Form('16:9'),
    background_color: str = Form('#ffffff'),
    caption_style: str = Form('tiktok'),
    title: str = Form('Chapter 1'),
    subtitle: str = Form('')
):
    """Render video with captions."""
    captions = json.loads(captions_json)
    
    result = render_video(
        audio_path=audio_path,
        captions=captions,
        aspect_ratio=aspect_ratio,
        background_color=background_color,
        caption_style=caption_style,
        title=title,
        subtitle=subtitle
    )
    
    return {
        'video_path': result['video_path'],
        'aspect_ratio': aspect_ratio,
        'status': 'success'
    }


@app.post('/api/video/render-full')
async def api_render_full_audiobook(
    audio_paths_json: str = Form(...),
    captions_json: str = Form('[]'),
    aspect_ratio: str = Form('16:9'),
    background_color: str = Form('#ffffff'),
    caption_style: str = Form('tiktok'),
    title: str = Form('Audiobook'),
    subtitle: str = Form('')
):
    """Render full audiobook video (all chapters)."""
    audio_paths = json.loads(audio_paths_json)
    captions = json.loads(captions_json)
    
    video_paths = []
    for i, audio_path in enumerate(audio_paths):
        chapter_captions = [c for c in captions if c.get('chapter') == i + 1]
        result = render_video(
            audio_path=audio_path,
            captions=chapter_captions,
            aspect_ratio=aspect_ratio,
            background_color=background_color,
            caption_style=caption_style,
            title=f'{title} - Chapter {i + 1}',
            subtitle=subtitle
        )
        video_paths.append(result['video_path'])
    
    # Concatenate all chapter videos
    final_output = WORK_DIR / f'full_audiobook_{uuid.uuid4()}.mp4'
    concat_result = concat_chapters(video_paths, str(final_output))
    
    return {
        'video_path': concat_result['video_path'],
        'total_chapters': len(audio_paths),
        'status': 'success'
    }


# ---- File Download ----

@app.get('/api/download/{file_type}/{filename}')
async def api_download(file_type: str, filename: str):
    """Download generated files."""
    file_path = WORK_DIR / filename
    if not file_path.exists():
        raise HTTPException(404, 'File not found')
    
    media_type = 'audio/wav' if file_type == 'audio' else 'video/mp4'
    return FileResponse(file_path, media_type=media_type, filename=filename)


# ---- Health Check ----

@app.get('/api/health')
async def health():
    return {
        'status': 'healthy',
        'services': {
            'tts': 'fish-audio-s2-pro',
            'video': 'remotion',
            'gpu': 'available'
        }
    }

In [ ]:
# Cell 7: Start API Server + Expose via ngrok
!pip install pyngrok

import threading
import uvicorn
from pyngrok import ngrok

# Kill any existing tunnels
ngrok.kill()

# Set authtoken from Kaggle secrets
# Go to Kaggle → notebook → Add Data → Secrets → Add NGROK_AUTH_TOKEN
NGROK_AUTH_TOKEN = os.environ.get('NGROK_AUTH_TOKEN')
if not NGROK_AUTH_TOKEN:
    raise ValueError(
        'NGROK_AUTH_TOKEN not set!\n'
        '1. Go to https://ngrok.com → Auth Token\n'
        '2. In Kaggle notebook sidebar → Secrets → Add secret\n'
        '3. Name: NGROK_AUTH_TOKEN, Value: your token'
    )

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Start API server in background thread
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8889)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print('API server started on port 8889')

# Create tunnel
public_url = ngrok.connect(8889)
print(f'\n========================================')
print(f'  VoiceForge Engine Public URL:')
print(f'  {public_url}')
print(f'========================================')
print(f'\nUpdate your backend KAGGLE_ENGINE_URL to:')
print(f'  {public_url}')

In [ ]:
# Cell 8: Server is running! Use the public URL above.
# To stop: restart the notebook kernel
print('VoiceForge Engine is live!')